# Option Orchestrator: End-to-End Rescue Pipeline Walkthrough

This walkthrough demonstrates the end-to-end decision-support pipeline for distressed multi-leg options positions:
1. **Position Intake & Greeks Analysis**: Calculating Black-Scholes Greeks and portfolio risk for an in-jeopardy spread.
2. **Candidate Plan Generation**: Proposing candidate adjustments (rolls, hedges, partial exits) structured under strict JSON schemas.
3. **Deterministic Safety Firewall**: Validating candidate proposals against account constraints (max loss tolerance, buying power limits, and defined-risk invariants) before presenting to the user.

In [1]:
import sys
import json
from pathlib import Path

# Add backend package to path
backend_path = Path.cwd().parent / 'backend' if Path.cwd().name == 'notebooks' else Path.cwd() / 'backend'
sys.path.insert(0, str(backend_path))

from shared.schemas import PositionSnapshot, OptionLeg, AccountConstraints, RescueCandidate
from analysis_layer.greeks import GreeksAnalyzer
from validator_layer.rules import PlanValidator

print('Environment ready. Core modules imported successfully.')

Environment ready. Core modules imported successfully.


## 1. Distressed Position Snapshot

**Scenario**: An SPX short put credit spread (4960 / 4920) was opened for a net credit of $0.90 per share ($90/contract).
- Underlying spot price fell from 5,060 to 5,000, rapidly approaching the 4960 short put strike.
- The position has 3 days until expiration, resulting in sharp negative gamma and expanding spread value (unrealized loss of ~88%).

In [2]:
# Define the distressed position legs
position = PositionSnapshot(
    symbol='SPX',
    legs=[
        OptionLeg(
            option_type='put',
            strike=4960.0,
            expiry='2026-02-06',
            quantity=1,
            side='short',
            entry_price=3.10,
        ),
        OptionLeg(
            option_type='put',
            strike=4920.0,
            expiry='2026-02-06',
            quantity=1,
            side='long',
            entry_price=1.40,
        ),
    ],
    notes='Short strike under pressure from spot selloff'
)

# Compute Greeks using Black-Scholes
analyzer = GreeksAnalyzer()
greeks = analyzer.calculate(position, spot_price=5000.0, implied_vol=0.25)

print('Position Snapshot: SPX Put Credit Spread (Short 4960P / Long 4920P)')
print('-' * 70)
print(f'Underlying Spot Price : ${5000.00:,.2f}')
print(f'Position Net Delta    : {greeks.delta:.4f}')
print(f'Position Net Gamma    : {greeks.gamma:.6f}')
print(f'Position Net Theta    : {greeks.theta:.2f} / day')
print(f'Position Net Vega     : {greeks.vega:.2f}')

Position Snapshot: SPX Put Credit Spread (Short 4960P / Long 4920P)
----------------------------------------------------------------------
Underlying Spot Price : $5,000.00
Position Net Delta    : -37.0864
Position Net Gamma    : 0.780413
Position Net Theta    : -1649.93 / day
Position Net Vega     : 133.63


## 2. Generating Candidate Rescue Plans

The domain-adapted Gemma 2 9B model receives the normalized position snapshot, implied volatility surface, and account risk limits. It outputs structured candidate adjustments:
- **Candidate 1 (Defensive Roll)**: Roll the vertical spread out 14 days and down 40 points (to 4920/4880) to restore strike distance.
- **Candidate 2 (High-Risk Financing)**: Sell a naked 4900 put to collect immediate premium. While tempting to naive models, this converts a defined-risk spread into naked short exposure.

In [3]:
candidate_valid = RescueCandidate(
    action='roll_out_and_down',
    rationale='Roll spread out 14 days and down 40 points to 4920/4880 to restore delta buffer.',
    net_cost=150.0,
    risk_rating='moderate',
)

candidate_invalid = RescueCandidate(
    action='sell_naked_short_put',
    rationale='Sell 1 naked 4900 put to collect $1.80 credit to offset current loss.',
    net_cost=-180.0,
    risk_rating='high',
)

print(f'Candidate 1: Roll Out & Down (4920/4880P, +14 DTE) | Net Cost: ${candidate_valid.net_cost:.2f} | Risk: {candidate_valid.risk_rating}')
print(f'Candidate 2: Sell Naked Put (4900P)                 | Net Cost: ${candidate_invalid.net_cost:.2f} | Risk: {candidate_invalid.risk_rating}')

Candidate 1: Roll Out & Down (4920/4880P, +14 DTE) | Net Cost: $150.00 | Risk: moderate
Candidate 2: Sell Naked Put (4900P)                 | Net Cost: -$180.00 | Risk: high


## 3. Deterministic Safety Firewall

The rule validator intercepts candidate proposals before they can be displayed. It enforces:
1. **Capital At Risk**: Net adjustment cost must not exceed max loss tolerance or available buying power.
2. **Risk Preference Invariants**: Conservative accounts reject `high` or `very_high` risk classifications.
3. **Defined-Risk Integrity**: Naked strategies are blocked unconditionally in non-aggressive accounts.

In [4]:
# Define conservative account parameters
account = AccountConstraints(
    buying_power=2500.0,
    max_loss_tolerance=500.0,
    risk_preference='conservative',
)

validator = PlanValidator()

print('Account Constraints:')
print(f'  Buying Power       : ${account.buying_power:,.2f}')
print(f'  Max Loss Tolerance : ${account.max_loss_tolerance:,.2f}')
print(f'  Risk Preference    : {account.risk_preference}')
print('-' * 70)

# Validate Candidate 1
res1 = validator.validate(candidate_valid, account)
print(f'Evaluating Candidate 1 [{candidate_valid.action}]:')
print(f'  Status  : {"APPROVED" if res1.valid else "REJECTED"}')
if res1.valid:
    print('  Action  : Candidate permitted under conservative risk guidelines.')
else:
    print(f'  Reason  : {res1.blocked_reason}')

# Validate Candidate 2
res2 = validator.validate(candidate_invalid, account)
print(f'\nEvaluating Candidate 2 [{candidate_invalid.action}]:')
print(f'  Status  : {"APPROVED" if res2.valid else "REJECTED"}')
if res2.valid:
    print('  Action  : Candidate permitted under conservative risk guidelines.')
else:
    print(f'  Reason  : {res2.blocked_reason}')

Account Constraints:
  Buying Power       : $2,500.00
  Max Loss Tolerance : $500.00
  Risk Preference    : conservative
----------------------------------------------------------------------
Evaluating Candidate 1 [roll_out_and_down]:
  Status  : APPROVED
  Action  : Candidate permitted under conservative risk guidelines.

Evaluating Candidate 2 [sell_naked_short_put]:
  Status  : REJECTED
  Reason  : risk_rating 'high' is not permitted under conservative risk preference; Naked strategies not permitted under conservative risk preference


## 4. Key Takeaways

- **Probabilistic AI handles strategy discovery**: The LLM suggests adjustments tailored to the specific position mechanics and volatility surface.
- **Deterministic code handles safety**: Hard rules prevent catastrophic failure modes (converting defined-risk into naked short legs, exceeding buying power, or inverting strikes).
- **No unattended trading**: Every plan is advisory, fully explained, and requires explicit user authorization before execution.